# Hooks Part 1: Pre-Tool-Use Logging

A hook is a Python function that the Claude Code application (not Claude itself) invokes at specific points in the agent loop. Hooks give you deterministic processing and automated feedback — observability and control that don't depend on the model deciding to cooperate.


In [2]:
from datetime import datetime
from typing import Any

from claude_agent_sdk import (
    tool,  # decorator that turns a Python function into a Claude-usable tool
    create_sdk_mcp_server,  # bundles one or more tools into a "server" Claude can talk to
    ClaudeSDKClient,  # a client you can keep open and send several messages through
    ClaudeAgentOptions,  # settings object: model, system prompt, tools, etc.
    HookMatcher,  # says WHICH tool a hook should watch
    HookContext,  # extra info passed into a hook function when it fires
    ResultMessage,  # the last message in the stream — carries the final answer plus stats (cost, duration, etc.)
)

# Same mock stock tool as earlier episodes — reused here just so we have
# a tool to attach a hook to.
MOCK_PRICES = {"AAPL": 193.50, "GOOGL": 178.25, "MSFT": 412.80}


def get_stock_price(ticker: str) -> dict[str, float]:
    """Mock stock price lookup — no real API call."""
    return {"price": MOCK_PRICES.get(ticker.upper(), 100.00)}


@tool("get_stock_price", "Get the current mock stock price for a ticker symbol", {"ticker": str})
async def get_stock_price_tool(args: dict[str, Any]) -> dict[str, Any]:
    price = get_stock_price(args["ticker"])
    return {"content": [{"type": "text", "text": f"{args['ticker']}: ${price['price']:.2f}"}]}


stock_server = create_sdk_mcp_server(name="stocks", version="1.0.0", tools=[get_stock_price_tool])

## Register a `PreToolUse` hook that logs before the tool runs


In [ ]:
call_log: list[dict[str, Any]] = []


# This is a "PreToolUse" hook function — it runs automatically, right BEFORE
# any matching tool call executes. It is not something Claude calls; the SDK
# itself calls it, guaranteed, every single time.
async def log_before_tool_use(
    input_data: dict[str, Any],  # info about the tool about to run (name + input)
    tool_use_id: str | None,  # unique id for this specific tool call
    context: HookContext,  # extra session context (not used here)
) -> dict[str, Any]:
    call_log.append(
        {
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "tool_name": input_data["tool_name"],
            "tool_input": input_data["tool_input"],
        }
    )
    return {}  # empty dict = allow the tool call to proceed unchanged
    # (a hook could also return data here to block or modify the tool call)


options = ClaudeAgentOptions(
    model="haiku",
    mcp_servers={"stocks": stock_server},
    allowed_tools=["mcp__stocks__get_stock_price"],
    # hooks: register our function to run at the "PreToolUse" point in the loop.
    # HookMatcher(matcher=..., hooks=[...]) means:
    #   "only fire these hook functions when the tool name matches this pattern"
    hooks={"PreToolUse": [HookMatcher(matcher="mcp__stocks__get_stock_price", hooks=[log_before_tool_use])]},
)

## Trigger the tool and inspect what the hook captured


In [7]:
async def run_with_hook() -> None:
    async with ClaudeSDKClient(options=options) as client:
        await client.query("What's the price of MSFT?")
        async for message in client.receive_response():
            if isinstance(message, ResultMessage):
                print(message.result)


await run_with_hook()

# call_log was filled in by our hook, not by us reading the messages —
# proof the hook really did fire on its own before the tool ran.
print("\n--- Pre-tool-use log (captured before the tool ran) ---")
for entry in call_log:
    print(entry)

Calling log_before_tool_use hook
The current price of MSFT (Microsoft) is **$412.80**.

--- Pre-tool-use log (captured before the tool ran) ---
{'timestamp': '2026-08-24T10:33:53', 'tool_name': 'mcp__stocks__get_stock_price', 'tool_input': {'ticker': 'MSFT'}}
{'timestamp': '2026-08-24T10:34:41', 'tool_name': 'mcp__stocks__get_stock_price', 'tool_input': {'ticker': 'MSFT'}}


## Summary

- Hooks fire deterministically at fixed points in the agent loop, independent of what the model decides.
- A `PreToolUse` hook gives you observability (or the power to block/modify) without touching the tool's own code.
